# Comprendre et tester la répétition des ouvertures sur Lichess

[▶ Ouvrir dans Colab](https://colab.research.google.com/github/Hadrienfonrojet/ANALYSE-MARCHE-/blob/main/Analyse_data.ipynb)

**Question** : le résultat précédent est-il associé à la répétition d’une ouverture ? La répétition est-elle associée au résultat suivant, selon la couleur, la cadence et le niveau Elo ?

Ce notebook contient le code essentiel des tests, pas seulement un appel caché à un script. Les explications sont volontairement détaillées : vous pouvez retirer les cellules de texte avant votre rendu. Exécuter dans l’ordre. Les sorties dépendent des données réellement collectées ; aucune conclusion Lichess n’est préécrite.


## Plan de travail
1. Installation et paramètres ; 2. Collecte et qualité ; 3. Variables temporelles ; 4. Descriptifs ; 5. Code du χ² ; 6. Quatre définitions de répétition ; 7. Tests et tableaux détaillés ; 8. Stratifications ; 9. Comparaison des définitions ; 10. Sensibilité et bootstrap ; 11. Corrélations et graphiques ; 12. Export et interprétation.

**Unité** : une observation est le point de vue d’un joueur sur une partie. Un même `game_id` peut apparaître pour deux joueurs. Une transition relie deux parties d’un joueur dans l’échantillon collecté ; elle n’est pas forcément consécutive dans son historique complet, car les requêtes sont limitées par cadence et par nombre de parties.


In [ ]:
# 1. Installation des bibliothèques (dont boto3) et récupération du projet
import os, sys, subprocess
from pathlib import Path
if not Path('Pipeline_ouvertures.py').exists():
    target = Path('/content/ANALYSE-MARCHE-') if Path('/content').exists() else Path('ANALYSE-MARCHE-')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Hadrienfonrojet/ANALYSE-MARCHE-.git', str(target)], check=True)
    os.chdir(target)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Projet :', Path.cwd())


## 1 Paramètres de collecte
Commencer petit pour vérifier la chaîne. Un petit échantillon rendra souvent les χ² inexploitables : augmenter le nombre de joueurs et de parties avant de discuter les valeurs p. `RAW_FILE` permet de réutiliser vos données sans nouvelles requêtes.


In [ ]:
SEEDS = ['yuuki-asuna']  # Remplacer par vos comptes ; cinq graines conseillées
USERS_PER_SEED = 1
GAMES_PER_FORMAT = 5
RAW_FILE = None  # Ou un chemin vers un fichier games_enrichies_raw.parquet existant
os.environ['LICHESS_SEEDS'] = ','.join(SEEDS)


## 3. Nouvelle expérience
Cette cellule ne supprime rien. La relancer crée une nouvelle expérience.


In [ ]:
import config_projet as cfg
cfg.SEED_USERS = [s.lower().strip() for s in SEEDS]
run_dir = cfg.create_run()
print('Dossier de cette expérience :', run_dir)


## 4. Collecte via l’API ou réutilisation de données
Les requêtes sont séquentielles ; une limitation Lichess peut entraîner une attente.


In [ ]:
import pandas as pd
from Scripts_generation_data.Collecte_enrichie import discover_stratified, collect_games
if RAW_FILE is None:
    token = cfg.get_lichess_token()
    users = discover_stratified(cfg.SEED_USERS, token, USERS_PER_SEED)
    raw = collect_games(users, token, GAMES_PER_FORMAT)
else:
    raw = pd.read_parquet(RAW_FILE)
    raw.to_parquet(cfg.DATA_DIR / 'games_enrichies_raw.parquet', index=False)
print('Observations brutes :', len(raw))
raw.head()


## 2 Qualité et couverture des données
Avant tout test, contrôler l’effectif, les valeurs manquantes et la part des ouvertures connues. Une famille inconnue ne doit pas être assimilée à une ouverture répétée. La collecte par adversaires n’est pas représentative de tous les joueurs de Lichess.


In [ ]:
display(pd.DataFrame({'manquants': raw.isna().sum(), 'part_manquante': raw.isna().mean()}))
print('Joueurs :', raw.user_id.nunique(), 'Parties distinctes :', raw.game_id.nunique())
print('Points de vue supplémentaires sur une même partie :', raw.game_id.duplicated().sum())
print('Période :', pd.to_datetime(raw.timestamp_ms, unit='ms', utc=True).min(), 'à', pd.to_datetime(raw.timestamp_ms, unit='ms', utc=True).max())


## 5. Ouvertures et nettoyage
Le référentiel contient uniquement les ouvertures observées dans les réponses de l’API.


In [ ]:
from Scripts_generation_data.Creation_openings_complet import build_openings
from Nettoyage_donnees import prepare
openings = build_openings(raw)
clean = prepare(raw, openings)
display(openings.head())
display(clean[['user_id', 'opening_family', 'color', 'rating', 'result']].head())


## 3 Trois comparaisons de couleur et deux historiques

- `repeat_all` : même famille que dans la partie observée immédiatement précédente, quelle que soit la couleur.
- `repeat_same_color` : même famille que dans la précédente partie observée avec la même couleur ; une ou plusieurs parties de l’autre couleur peuvent s’intercaler.
- `same_color_consecutive` : même couleur que dans la partie observée immédiatement précédente. Ce n’est pas une mesure de répétition d’ouverture.

Le croisement de la dernière variable avec `repeat_all` donne deux sous-populations : couleur répétée ou couleur changée. Les premières observations et les transitions avec une ouverture inconnue sont manquantes, jamais codées comme des changements. Le résultat précédent est calculé par `shift()` après tri, sans utiliser le résultat futur.


In [ ]:
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests
from pathlib import Path
def add_transitions(frame):
    """Comparer l'ouverture à la partie précédente, puis à la précédente de même couleur.

    Une ouverture inconnue ne constitue jamais une répétition ni un changement.
    L'historique porte sur l'échantillon collecté, pas sur toutes les parties Lichess.
    """
    d = frame.sort_values(['user_id', 'timestamp', 'game_id']).copy()
    known = d['opening_family'].notna() & ~d['opening_family'].isin(['Inconnue', '', 'UNKNOWN'])
    d['known_opening'] = known
    for suffix, keys in [('all', ['user_id']), ('same_color', ['user_id', 'color'])]:
        grouped = d.groupby(keys, observed=True)
        d['prev_game_' + suffix] = grouped['game_id'].shift()
        d['prev_opening_' + suffix] = grouped['opening_family'].shift()
        d['prev_result_' + suffix] = grouped['result'].shift()
        previous_known = grouped['known_opening'].shift().eq(True)
        valid = known & previous_known
        d['repeat_' + suffix] = np.where(
            valid, d['opening_family'].eq(d['prev_opening_' + suffix]).astype(float), np.nan)
    grouped = d.groupby('user_id', observed=True)
    d['prev_color'] = grouped['color'].shift()
    d['same_color_consecutive'] = np.where(d['prev_color'].notna(),
        d['color'].eq(d['prev_color']).astype(float), np.nan)
    return d


### Exemple lisible à la main
Pour blanc A, noir B, blanc A, blanc B : la troisième partie répète A à même couleur, mais pas par rapport à la partie précédente. C’est la différence essentielle entre les deux définitions.


In [ ]:
example = pd.DataFrame({'user_id':['exemple']*4, 'game_id':['e1','e2','e3','e4'], 'timestamp':pd.date_range('2024-01-01', periods=4, tz='UTC'), 'color':['white','black','white','white'], 'opening_family':['A','B','A','B'], 'result':['Win','Loss','Draw','Win']})
display(add_transitions(example)[['game_id','color','opening_family','repeat_all','repeat_same_color','same_color_consecutive']])
clean = add_transitions(clean)


## 4 Description des ouvertures et des résultats
Comparer effectifs et scores moyens. Le score moyen tient compte des nulles, contrairement au taux de victoire seul. La famille regroupe plusieurs variantes et dépend des choix des deux joueurs.


In [ ]:
description = clean.groupby(['opening_family', 'color'], observed=True).agg(
    parties=('game_id', 'size'), score_moyen=('score', 'mean'), taux_victoire=('win', 'mean')
).sort_values('parties', ascending=False)
display(description)


In [ ]:
display(pd.crosstab(clean['color'], clean['result']))
display(pd.crosstab(clean['format_partie'], clean['result'], normalize='index'))
display(clean.groupby('user_id').agg(parties=('game_id','size'),score=('score','mean'),familles=('opening_family','nunique')))


## 5 Le test du χ² est codé ici

**H0** : les deux variables catégorielles sont indépendantes dans la population étudiée. **H1** : elles sont associées. Pour chaque case, l’effectif attendu vaut `total_ligne × total_colonne / n`. Le χ² de Pearson vaut `Σ (observé − attendu)² / attendu` ; les degrés de liberté valent `(lignes − 1) × (colonnes − 1)`.

Le V de Cramér est `sqrt(χ² / [n × min(lignes−1, colonnes−1)])`. Il décrit l’intensité de l’association et n’a pas de signe. Le calcul ci-dessous désactive explicitement la correction de Yates pour rester cohérent avec cette formule.

La règle conservatrice retenue est **tous les effectifs attendus au moins égaux à 5**. `valid=True` valide seulement cette règle, pas l’indépendance des observations. Pour un tableau 2×2, Fisher est aussi calculé ; il ne résout pas la dépendance entre parties. Un tableau dégénéré retourne une valeur p manquante.


In [ ]:
def chi2_details(frame, a, b, label, min_expected=5):
    """Test de Pearson sans correction de Yates ; tableaux et diagnostic explicites."""
    complete = frame[[a, b]].dropna()
    observed = pd.crosstab(complete[a], complete[b])
    observed = observed.loc[observed.sum(axis=1).gt(0), observed.sum(axis=0).gt(0)]
    n = int(observed.to_numpy().sum())
    result = dict(test=label, n=n, chi2=np.nan, ddl=np.nan, p=np.nan,
                  cramer_v=np.nan, expected_lt5=np.nan, min_expected=np.nan,
                  valid=False, independence_assumed=True, interpretation='exploratoire',
                  fisher_p=np.nan, status='tableau insuffisant')
    expected = pd.DataFrame(index=observed.index, columns=observed.columns, dtype=float)
    residuals = expected.copy()
    if min(observed.shape, default=0) < 2:
        return result, observed, expected, residuals
    stat, p, ddl, exp = chi2_contingency(observed, correction=False)
    expected = pd.DataFrame(exp, index=observed.index, columns=observed.columns)
    residuals = (observed - expected) / np.sqrt(expected)
    result.update(chi2=float(stat), p=float(p), ddl=int(ddl),
        cramer_v=float(np.sqrt(stat / (n * min(observed.shape[0]-1, observed.shape[1]-1)))),
        expected_lt5=int((exp < min_expected).sum()), min_expected=float(exp.min()),
        valid=bool((exp >= min_expected).all()),
        status='effectifs suffisants' if (exp >= min_expected).all() else 'effectifs faibles')
    if observed.shape == (2, 2):
        result['fisher_p'] = float(fisher_exact(observed.to_numpy())[1])
    return result, observed, expected, residuals


### Hypothèses et plan de tests
Pour chaque définition de répétition : (A) résultat précédent × répétition ; (B) répétition × résultat courant. On ajoute les analyses par couleur, cadence et tranche Elo, puis la répétition de la couleur elle-même. Les stratifications comparent des sous-populations mais ne démontrent pas qu’un effet diffère statistiquement entre elles. Une petite valeur p dans un groupe et une grande dans l’autre ne suffisent pas à conclure à une interaction.


In [ ]:
def repetition_scenarios(frame):
    """Définir les populations avant de lancer les tests."""
    d = add_transitions(frame)
    return {
        'toutes_couleurs': (d, 'repeat_all', 'prev_result_all', 'prev_game_all'),
        'precedente_meme_couleur': (d, 'repeat_same_color', 'prev_result_same_color', 'prev_game_same_color'),
        'consecutives_meme_couleur': (d[d['same_color_consecutive'].eq(1)], 'repeat_all', 'prev_result_all', 'prev_game_all'),
        'consecutives_couleur_changee': (d[d['same_color_consecutive'].eq(0)], 'repeat_all', 'prev_result_all', 'prev_game_all'),
    }


## 6 Fonctions de sensibilité et multiplicité
Le filtrage de sensibilité garde au plus une transition par joueur et évite les identifiants de parties partagés entre transitions. Le prix est une forte réduction des effectifs ; les adversaires communs peuvent encore créer une dépendance.

La correction de Holm est appliquée à l’ensemble des χ² ayant des effectifs suffisants dans cette exécution. Elle contrôle le risque familial sous des tests dépendants si leurs valeurs p sont valides. Elle ne répare pas la dépendance entre observations ni un mauvais plan d’échantillonnage. Les valeurs p de Fisher ne font pas partie de cette famille corrigée et restent des diagnostics.


In [ ]:
def correct_multiple_tests(results):
    """Holm contrôle le risque familial sans exiger des tests indépendants.

    Cela ne répare pas les valeurs p invalides dues aux observations dépendantes.
    """
    out = results.copy()
    out['p_holm'] = np.nan
    usable = out['p'].notna() & out['valid'].fillna(False)
    if usable.any():
        out.loc[usable, 'p_holm'] = multipletests(out.loc[usable, 'p'], method='holm')[1]
    return out

def independent_transition_sample(frame, previous_game):
    """Sensibilité : au plus une transition par joueur et aucun ID de partie partagé.

    Ce filtrage ne supprime pas les adversaires communs ou les autres dépendances.
    """
    used_games, used_users, chosen = set(), set(), []
    ordered = frame.dropna(subset=[previous_game]).sort_values(['timestamp', 'game_id', 'user_id'])
    for idx, row in ordered.iterrows():
        games = {str(row['game_id']), str(row[previous_game])}
        if row['user_id'] not in used_users and not games & used_games:
            chosen.append(idx)
            used_games.update(games)
            used_users.add(row['user_id'])
    return frame.loc[chosen].copy()


### Bootstrap des joueurs
Pour chaque joueur ayant à la fois répétitions et changements, calculer `score moyen en répétition − score moyen en changement`, puis moyenner avec un poids égal par joueur. Rééchantillonner ces différences donne un intervalle descriptif à 95 %. Il ne s’agit ni d’un effet causal, ni du même estimand que le score global par partie. Les adversaires communs et les joueurs sans les deux situations restent des limites.


In [ ]:
def player_bootstrap(frame, repeat_column, n_boot=1000, seed=42):
    """IC descriptif du contraste de score, pondération égale des joueurs éligibles.

    Rééchantillonnage des joueurs ayant répétition ET changement ; ni causalité,
    ni correction complète de la dépendance entre adversaires communs.
    """
    valid = frame.dropna(subset=[repeat_column, 'score'])
    means = valid.groupby(['user_id', repeat_column], observed=True)['score'].mean().unstack()
    if 0.0 not in means or 1.0 not in means:
        return dict(players=0, difference=np.nan, ci_low=np.nan, ci_high=np.nan)
    delta = (means[1.0] - means[0.0]).dropna().to_numpy()
    if len(delta) < 2:
        return dict(players=len(delta), difference=float(delta.mean()) if len(delta) else np.nan,
                    ci_low=np.nan, ci_high=np.nan)
    rng = np.random.default_rng(seed)
    draws = np.array([rng.choice(delta, size=len(delta), replace=True).mean() for _ in range(n_boot)])
    return dict(players=len(delta), difference=float(delta.mean()),
                ci_low=float(np.quantile(draws, .025)), ci_high=float(np.quantile(draws, .975)))


## 7 Tests explicites et affichage des effectifs
Voici l’appel direct au test, avant la batterie complète. H0 : la répétition à même couleur et le résultat courant sont indépendants. La table attendue et les résidus de Pearson permettent de comprendre le χ². Les résidus sont descriptifs ; les cases ne sont pas testées séparément.


In [ ]:
resultat, observes, attendus, residus = chi2_details(clean, 'repeat_same_color', 'result', 'repetition_meme_couleur_resultat')
display(pd.Series(resultat))
print('Effectifs observés'); display(observes)
print('Effectifs attendus sous H0'); display(attendus)
print('Résidus de Pearson'); display(residus)


### Batterie complète avec quatre définitions
Le code ci-dessous définit chaque test puis exporte un CSV pour les observations, les attentes et les résidus. L’index `index_tables.csv` fait le lien entre numéro de table et nom du test. Les données brutes ne sont pas modifiées.


In [ ]:
def run_repetition_tests(frame, output_dir=None):
    """Deux questions par définition, puis stratifications, sensibilité et exports."""
    records, tables, rates, boot = [], {}, [], []
    scenarios = repetition_scenarios(frame)
    def run(part, a, b, name):
        record, observed, expected, residuals = chi2_details(part, a, b, name)
        records.append(record)
        tables[name] = (observed, expected, residuals)
    for name, (part, repeat, previous, previous_game) in scenarios.items():
        eligible = part.dropna(subset=[repeat, previous, 'result'])
        run(eligible, previous, repeat, name + '__resultat_precedent_repetition')
        run(eligible, repeat, 'result', name + '__repetition_resultat_courant')
        rates.append(dict(scenario=name, transitions=len(eligible),
            players=eligible['user_id'].nunique(),
            repeat_rate=float(eligible[repeat].mean()) if len(eligible) else np.nan))
        boot.append(dict(scenario=name, **player_bootstrap(eligible, repeat)))
        sensitivity = independent_transition_sample(eligible, previous_game)
        run(sensitivity, repeat, 'result', name + '__sensibilite_une_transition')
        for dim in ['color', 'format_partie', 'elo_band']:
            for value, subset in eligible.groupby(dim, observed=True):
                run(subset, repeat, 'result', name + '__' + dim + '_' + str(value))
    d = add_transitions(frame)
    run(d, 'same_color_consecutive', 'result', 'couleur_repetee__resultat_courant')
    run(d, 'same_color_consecutive', 'repeat_all', 'couleur_repetee__ouverture_repetee')
    results = correct_multiple_tests(pd.DataFrame(records))
    rates, boot = pd.DataFrame(rates), pd.DataFrame(boot)
    if output_dir is not None:
        out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
        results.to_csv(out / 'tests_repetition_chi2.csv', index=False)
        rates.to_csv(out / 'taux_repetition.csv', index=False)
        boot.to_csv(out / 'bootstrap_joueurs.csv', index=False)
        for i, (name, values) in enumerate(tables.items()):
            for kind, tab in zip(['observes', 'attendus', 'residus'], values):
                tab.to_csv(out / f'table_{i:03d}_{kind}.csv')
        pd.DataFrame({'table_id': range(len(tables)), 'test': list(tables)}).to_csv(out / 'index_tables.csv', index=False)
    return results, tables, rates, boot


In [ ]:
tests_repetition, tables, taux, bootstrap = run_repetition_tests(clean, cfg.RESULTS_DIR / 'repetition')
columns = ['test','n','chi2','ddl','p','p_holm','cramer_v','min_expected','valid','status']
display(tests_repetition[columns])
print('Tests à effectifs suffisants :', tests_repetition['valid'].sum(), '/', len(tests_repetition))


## 8 Résultat précédent et répétition
H0 : rejouer une famille est indépendant du résultat précédent. Examiner victoires, défaites et nulles séparément. Le résultat précédent utilisé doit correspondre au même historique que la répétition : précédente partie toutes couleurs ou précédente de même couleur.


In [ ]:
selection = tests_repetition['test'].str.endswith('__resultat_precedent_repetition')
display(tests_repetition.loc[selection, columns])
for name, (part, repeat, previous, _) in repetition_scenarios(clean).items():
    print(name)
    display(pd.crosstab(part[previous], part[repeat], normalize='index'))


## 9 Couleur répétée ou couleur changée
H0 des deux tests ci-dessous : (1) répétition de couleur et résultat courant sont indépendants ; (2) répétition de couleur et répétition d’ouverture sont indépendantes. Ces questions diffèrent de la répétition d’une ouverture à même couleur.


In [ ]:
display(tests_repetition.loc[tests_repetition['test'].str.startswith('couleur_repetee__'), columns])
display(pd.crosstab(clean['same_color_consecutive'], clean['repeat_all']))
display(pd.crosstab(clean['same_color_consecutive'], clean['result'], normalize='index'))


## 10 Analyses stratifiées par couleur cadence et Elo
Les préférences d’ouverture et les résultats dépendent du niveau, de la cadence et de la couleur. Ces tableaux cherchent une stabilité descriptive des associations. Ne pas multiplier ou fusionner les catégories après lecture des valeurs p pour obtenir un résultat significatif.


In [ ]:
for dimension in ['color','format_partie','elo_band']:
    print('Stratification :', dimension)
    display(tests_repetition.loc[tests_repetition['test'].str.contains('__'+dimension+'_', regex=False), columns])


## 11 Comparer les taux de répétition
Les quatre définitions n’ont pas le même dénominateur. Afficher le nombre de transitions et de joueurs avant de comparer les taux. Les taux ne constituent pas un test de différence entre définitions.


In [ ]:
display(taux)
import matplotlib
from IPython import get_ipython
if get_ipython() is None:
    matplotlib.use('Agg')
else:
    get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,4))
ax.barh(taux['scenario'], taux['repeat_rate'])
ax.set(xlim=(0,1), xlabel='Proportion de transitions répétées', title='Répétition selon la définition')
fig.tight_layout(); fig.savefig(cfg.RESULTS_DIR/'taux_repetition.png', dpi=150)
display(fig); plt.close(fig)


### Matrice de transition des résultats
Lire chaque ligne conditionnellement au résultat précédent. Les transitions se chevauchent et peuvent être corrélées : cette matrice décrit une dynamique sans identifier un mécanisme causal.


In [ ]:
display(pd.crosstab(clean['prev_result_all'], clean['result'], normalize='index'))
family_counts = clean.loc[clean['known_opening'], 'opening_family'].value_counts()
top_families = family_counts.head(8).index
pairs = clean[clean['opening_family'].isin(top_families) & clean['prev_opening_all'].isin(top_families)]
display(pd.crosstab(pairs['prev_opening_all'], pairs['opening_family'], normalize='index'))


## 12 Contrôles de sensibilité
Afficher les résultats avec une transition maximum par joueur et sans partie partagée. Si les effectifs attendus deviennent trop faibles, la bonne conclusion est une information insuffisante, pas l’absence d’association. Les deux points de vue d’une partie ne sont jamais deux parties indépendantes.


In [ ]:
display(tests_repetition.loc[tests_repetition['test'].str.endswith('__sensibilite_une_transition'), columns])
display(bootstrap)


### Pondération par joueur
Comparer l’analyse par partie à une moyenne des taux individuels. Le second calcul donne autant de poids à un joueur peu actif qu’à un joueur très actif ; il répond donc à une autre question.


In [ ]:
by_player = clean.groupby('user_id').agg(parties=('game_id','size'), repetition=('repeat_all','mean'),score=('score','mean'))
display(by_player)
print('Taux moyen par joueur :', by_player.repetition.mean())
print('Taux global par transition :', clean.repeat_all.mean())


## 13 Corrélations de Spearman
Une corrélation de rang résume une association monotone. L’expérience antérieure est calculée avant la partie et seulement dans la fenêtre collectée. Elle n’est pas une mesure complète de l’expérience échiquéenne. Les variables dérivées peuvent être mécaniquement corrélées : `elo_diff = rating − opponent_rating`.


In [ ]:
import matplotlib.pyplot as plt
columns = ['rating', 'opponent_rating', 'elo_diff', 'experience_opening_prior', 'score']
corr = clean[columns].corr(method='spearman')
display(corr)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(columns)), columns, rotation=45, ha='right')
ax.set_yticks(range(len(columns)), columns)
fig.colorbar(im, ax=ax, label='Spearman')
fig.tight_layout()
fig.savefig(cfg.RESULTS_DIR / 'correlations_spearman.png', dpi=150)
display(fig)
plt.close(fig)


## 14 Scores par famille et expérience
Lire ces courbes avec leurs effectifs. Une progression apparente peut provenir de la sélection des joueurs, d’adversaires différents ou d’une modification de cadence.


In [ ]:
summary = clean.groupby('opening_family').agg(parties=('score', 'size'), score=('score', 'mean')).nlargest(12, 'parties')
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
summary['parties'].plot.barh(ax=axes[0], title='Effectifs par famille')
summary['score'].plot.barh(ax=axes[1], title='Score moyen par famille', xlim=(0, 1))
fig.tight_layout()
fig.savefig(cfg.RESULTS_DIR / 'ouvertures_scores.png', dpi=150)
display(fig)
plt.close(fig)


In [ ]:
experience = clean.copy()
experience['classe_experience'] = pd.cut(experience.experience_opening_prior, [-1,0,1,4,9,1000000], labels=['première','deuxième','3 à 5','6 à 10','11 et plus'])
display(experience.groupby(['classe_experience','color'], observed=True).agg(n=('score','size'),score=('score','mean')))


## 15 Analyses générales et exports complémentaires
Le moteur historique est maintenant dans `analyses/Analyse_ouvertures.py`. Les régressions ne sont pas exécutées par défaut ; l’essentiel du projet porte sur contingences, χ², tailles d’association et descriptifs.


In [ ]:
from Analyse_ouvertures import analyze
tests_generaux = analyze(clean)
display(tests_generaux)


## 16 Rédiger une interprétation justifiée
Pour chaque résultat retenu, préciser : population et période ; définition de répétition ; tableau et effectif ; χ², degrés de liberté, valeur p et correction ; V de Cramér ; effectifs attendus ; limites de dépendance. Écrire « associé à », et non « provoque ».

Exemple de structure à compléter après exécution : « Dans cet échantillon, selon la définition …, le tableau contient … transitions. Le test présente … ; les effectifs attendus sont … . La taille de l’association est … . La conclusion reste exploratoire en raison de … . » Ne pas remplir ces blancs avec les données simulées des tests logiciels.


## 10. Résultats et sauvegarde
Dans Colab, cette cellule télécharge les données et résultats de l’expérience. Elle n’envoie rien vers SSP Cloud. Les fichiers générés ne sont pas ajoutés automatiquement au dépôt GitHub.


In [ ]:
import shutil
print('Résultats :', cfg.RESULTS_DIR)
archive = shutil.make_archive(str(run_dir), 'zip', run_dir)
print('Archive :', archive)
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(archive))
else:
    files.download(archive)


## 11. SSP Cloud — envoi optionnel
`boto3` est installé dans la première cellule. Renseigner les identifiants uniquement dans l’environnement privé. Passer `UPLOAD` à `True` pour envoyer cette expérience sous un préfixe unique.


In [ ]:
UPLOAD = False
if UPLOAD:
    from Stockage_SSPCloud import upload_outputs
    upload_outputs()


## 17 Où se trouvent les fonctions
- Les cellules de ce notebook montrent le code de `add_transitions`, `chi2_details`, `correct_multiple_tests`, `repetition_scenarios`, `independent_transition_sample`, `player_bootstrap` et `run_repetition_tests`.
- Leur version de référence se trouve dans `analyses/Tests_repetition.py` ; le notebook contient la même version lors de cette publication.
- `prepare` est dans `Nettoyage_donnees.py` et le moteur général dans `analyses/Analyse_ouvertures.py`.
- Les tests logiciels sont dans `tests/test_analyses.py` ; les tests statistiques produisent `Resultats/repetition/tests_repetition_chi2.csv`.

## Références techniques
[SciPy χ²](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html) ; [Fisher](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.fisher_exact.html) ; [correction multiple statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html).
